# RT-DETR UI Element Detection Training on Kaggle GPU

This notebook fine-tunes the RT-DETR-R18 object detection model on a massive merged UI dataset (~31,000 images):
1. **ScreenSpot (All Splits):** Mobile, Desktop, and Web (2,853 samples)
2. **GroundUI-18k:** Cross-platform GUI elements (18,026 samples)
3. **Rico Mobile UI:** Android app layouts (10,000 sample subset)

### Instructions:
1. In Kaggle, go to **Settings** (right sidebar) and verify that **Accelerator** is set to **GPU T4 x2** or **GPU T4 x1**.
2. Toggle the **Internet on** setting (required to download the datasets from Hugging Face).
3. Click **Run All** to run all cells.
4. Download the final zipped model checkpoint (`rtdetr-ui-complete-checkpoint-best.zip`) from the Output section.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers accelerate torchvision huggingface_hub pycocotools safetensors fiftyone

## 1.5. (Optional) Log in to Hugging Face Hub

To avoid rate limits during downloading of the datasets, you can add your Hugging Face token as a Kaggle User Secret:
1. Go to **Add-ons** > **Secrets** in the top menu of the Kaggle editor.
2. Add a new secret with Label: `HF_TOKEN` and Value: your Hugging Face API token.
3. Click **Save** and verify that the toggle for the secret is checked.

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Successfully logged into Hugging Face Hub!")
    else:
        print("HF_TOKEN secret not found in Kaggle Secrets. Continuing with unauthenticated requests.")
except Exception as e:
    print("Could not load Kaggle UserSecretsClient or login failed. Continuing with unauthenticated requests.")

## 2. Write Dataset Preparation Script

In [ ]:
%%writefile prepare_mobile_mixed_coco.py
import json
import random
from dataclasses import dataclass
from pathlib import Path
import zipfile
from typing import Any
from huggingface_hub import hf_hub_download, snapshot_download
from PIL import Image

PRO_DATASET_ID = "likaixin/ScreenSpot-Pro"
V2_DATASET_ID = "OS-Copilot/ScreenSpot-v2"
SUPPORTED_CLASSES = ("icon", "text")

@dataclass(frozen=True)
class Record:
    source: str
    image_rel: str
    ui_type: str
    bbox: tuple[float, float, float, float]
    img_size: tuple[int, int] | None

def _load_pro_records(raw_root: Path) -> list[Record]:
    ann_dir = raw_root / "annotations"
    files = sorted(ann_dir.glob("*.json"))
    records = []
    for ann_file in files:
        with ann_file.open("r", encoding="utf-8") as f:
            items = json.load(f)
        for item in items:
            ui_type = str(item.get("ui_type", "")).strip().lower()
            if ui_type not in SUPPORTED_CLASSES:
                continue
            image_rel = str(item.get("img_filename", "")).strip()
            bbox_raw = item.get("bbox", [])
            if len(bbox_raw) != 4 or not image_rel:
                continue
            try:
                bbox = (float(bbox_raw[0]), float(bbox_raw[1]), float(bbox_raw[2]), float(bbox_raw[3]))
            except (TypeError, ValueError):
                continue
            size = item.get("img_size")
            img_size = None
            if isinstance(size, list) and len(size) == 2:
                try:
                    img_size = (int(size[0]), int(size[1]))
                except (TypeError, ValueError):
                    img_size = None
            records.append(Record(source="pro", image_rel=image_rel, ui_type=ui_type, bbox=bbox, img_size=img_size))
    return records

def _load_v2_records(raw_root: Path, filename: str, source_name: str) -> list[Record]:
    ann_file = raw_root / filename
    with ann_file.open("r", encoding="utf-8") as f:
        items = json.load(f)
    records = []
    for item in items:
        ui_type = str(item.get("data_type", "")).strip().lower()
        if ui_type not in SUPPORTED_CLASSES:
                continue
        image_rel = str(item.get("img_filename", "")).strip()
        bbox_raw = item.get("bbox", [])
        if len(bbox_raw) != 4 or not image_rel:
            continue
        try:
            bbox = (float(bbox_raw[0]), float(bbox_raw[1]), float(bbox_raw[2]), float(bbox_raw[3]))
        except (TypeError, ValueError):
            continue
        records.append(Record(source=source_name, image_rel=image_rel, ui_type=ui_type, bbox=bbox, img_size=None))
    return records

def _bbox_to_xywh(bbox: tuple[float, float, float, float]) -> tuple[float, float, float, float]:
    x1, y1, x2, y2 = bbox
    if x2 > x1 and y2 > y1:
        return x1, y1, x2 - x1, y2 - y1
    return x1, y1, x2, y2

def _xywh_from_record(rec: Record) -> tuple[float, float, float, float]:
    if rec.source in ("mobile_v2", "desktop_v2", "web_v2"):
        return rec.bbox
    return _bbox_to_xywh(rec.bbox)

def _build_mobile_image_index(extract_root: Path) -> dict[str, str]:
    index = {}
    for path in extract_root.rglob("*.png"):
        index[path.name] = str(path)
    return index

def _ensure_mobile_v2_images(out_root: Path, needed_names: set[str]) -> dict[str, str]:
    if not needed_names:
        return {}
    raw_root = out_root / "raw" / "screenspot_v2"
    raw_root.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id=V2_DATASET_ID, repo_type="dataset", filename="screenspotv2_image.zip", local_dir=str(raw_root))
    extract_root = raw_root / "images"
    extract_root.mkdir(parents=True, exist_ok=True)
    index = _build_mobile_image_index(extract_root)
    missing = {name for name in needed_names if name not in index}
    if missing:
        with zipfile.ZipFile(zip_path, "r") as zf:
            members = zf.namelist()
            member_lookup = {Path(m).name: m for m in members}
            for name in missing:
                member = member_lookup.get(name)
                if member is not None:
                    zf.extract(member, path=extract_root)
        index = _build_mobile_image_index(extract_root)
    return index

def _clamp_bbox(x: float, y: float, w: float, h: float, width: int, height: int) -> tuple[float, float, float, float] | None:
    x = max(0.0, min(x, float(width - 1)))
    y = max(0.0, min(y, float(height - 1)))
    w = max(1.0, min(w, float(width) - x))
    h = max(1.0, min(h, float(height) - y))
    return (x, y, w, h) if w > 0.0 and h > 0.0 else None

def _build_split(split_name: str, split_records: list[Record], out_root: Path, mobile_index: dict[str, str]) -> None:
    images_dir = out_root / split_name / "images"
    images_dir.mkdir(parents=True, exist_ok=True)
    coco = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "icon"}, {"id": 2, "name": "text"}]}
    image_id, annotation_id, skipped = 1, 1, 0
    for rec in split_records:
        if rec.source in ("mobile_v2", "desktop_v2", "web_v2"):
            local_file = mobile_index.get(rec.image_rel)
            if local_file is None:
                skipped += 1
                continue
        else:
            local_file = hf_hub_download(repo_id=PRO_DATASET_ID, repo_type="dataset", filename=f"images/{rec.image_rel}", local_dir=str(out_root / "raw" / "screenspot_pro"))
        local_name = f"{rec.source}__{rec.image_rel.replace('/', '__')}"
        dst_file = images_dir / local_name
        if not dst_file.exists():
            with Image.open(local_file) as source_img:
                source_img.save(dst_file)
        with Image.open(dst_file) as img:
            width, height = img.size
        coords = _xywh_from_record(rec)
        clamped = _clamp_bbox(coords[0], coords[1], coords[2], coords[3], width, height)
        if clamped is None:
            continue
        x, y, w, h = clamped
        coco["images"].append({"id": image_id, "file_name": local_name, "width": width, "height": height})
        coco["annotations"].append({"id": annotation_id, "image_id": image_id, "category_id": 1 if rec.ui_type == "icon" else 2, "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)], "area": round(w * h, 2), "iscrowd": 0})
        image_id += 1
        annotation_id += 1
    ann_file = out_root / split_name / "annotations.json"
    with ann_file.open("w", encoding="utf-8") as f:
        json.dump(coco, f)
    print(f"{split_name}: images={len(coco['images'])} annotations={len(coco['annotations'])} skipped={skipped}")

def main():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-root", type=Path, default=Path("data/screenspot"))
    parser.add_argument("--total-samples", type=int, default=-1)
    parser.add_argument("--val-ratio", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    out_root = args.output_root
    snapshot_download(repo_id=PRO_DATASET_ID, repo_type="dataset", local_dir=str(out_root / "raw" / "screenspot_pro"), allow_patterns=["annotations/*.json", "README.md", "eval.yaml"])
    v2_splits = {"mobile_v2": "screenspot_mobile_v2.json", "desktop_v2": "screenspot_desktop_v2.json", "web_v2": "screenspot_web_v2.json"}
    for filename in v2_splits.values():
        hf_hub_download(repo_id=V2_DATASET_ID, repo_type="dataset", filename=filename, local_dir=str(out_root / "raw" / "screenspot_v2"))
    pro_records = _load_pro_records(out_root / "raw" / "screenspot_pro")
    v2_records = []
    for source_name, filename in v2_splits.items():
        v2_records.extend(_load_v2_records(out_root / "raw" / "screenspot_v2", filename, source_name))
    if args.total_samples <= 0:
        sampled = pro_records + v2_records
        random.Random(args.seed).shuffle(sampled)
    else:
        # Fallback to simple random sample
        combined = pro_records + v2_records
        random.Random(args.seed).shuffle(combined)
        sampled = combined[:args.total_samples]
    # Split
    train_count = int(len(sampled) * (1 - args.val_ratio))
    train_records = sampled[:train_count]
    val_records = sampled[train_count:]
    needed_v2 = {r.image_rel for r in sampled if r.source in v2_splits}
    v2_index = _ensure_mobile_v2_images(out_root, needed_names=needed_v2)
    _build_split("train", train_records, out_root, mobile_index=v2_index)
    _build_split("val", val_records, out_root, mobile_index=v2_index)
    print(f"done: total={len(sampled)} train={len(train_records)} val={len(val_records)}")

if __name__ == '__main__':
    main()


## 3. Write Master Dataset Preparation & Merge Script

In [ ]:
%%writefile prepare_all_datasets.py
import os
import json
import shutil
import random
from pathlib import Path
import fiftyone as fo
import fiftyone.utils.huggingface as fouh

print("=== 1. Preparing ScreenSpot Suite ===")
os.system("python prepare_mobile_mixed_coco.py --output-root data/screenspot --total-samples -1 --val-ratio 0.15")

output_dir = Path("data/merged_coco")
train_img_dir = output_dir / "train" / "images"
val_img_dir = output_dir / "val" / "images"
train_img_dir.mkdir(parents=True, exist_ok=True)
val_img_dir.mkdir(parents=True, exist_ok=True)

merged_train = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "icon"}, {"id": 2, "name": "text"}]}
merged_val = {"images": [], "annotations": [], "categories": [{"id": 1, "name": "icon"}, {"id": 2, "name": "text"}]}

img_idx = 1
ann_idx = 1

def add_coco_dataset(img_src_dir, ann_path, val_ratio=0.15, dataset_prefix="dataset"):
    global img_idx, ann_idx
    with open(ann_path, "r") as f:
        data = json.load(f)
    
    cat_map = {}
    for cat in data["categories"]:
        name = cat["name"].lower()
        if "text" in name or "label" in name or "heading" in name:
            cat_map[cat["id"]] = 2
        else:
            cat_map[cat["id"]] = 1

    img_anns = {}
    for ann in data["annotations"]:
        img_anns.setdefault(ann["image_id"], []).append(ann)

    images = data["images"]
    # Seeded shuffle for reproducibility
    random.Random(42).shuffle(images)
    
    val_count = int(len(images) * val_ratio)
    val_images = set(img["id"] for img in images[:val_count])

    for img in images:
        old_id = img["id"]
        filename = img["file_name"]
        src_path = Path(img_src_dir) / filename
        if not src_path.exists():
            continue
            
        is_val = old_id in val_images
        dst_dir = val_img_dir if is_val else train_img_dir
        merged_data = merged_val if is_val else merged_train
        
        new_filename = f"{dataset_prefix}_{filename}"
        shutil.copy(src_path, dst_dir / new_filename)
        
        merged_data["images"].append({
            "id": img_idx,
            "file_name": new_filename,
            "width": img["width"],
            "height": img["height"]
        })
        
        anns = img_anns.get(old_id, [])
        for ann in anns:
            new_cat = cat_map.get(ann["category_id"], 1)
            merged_data["annotations"].append({
                "id": ann_idx,
                "image_id": img_idx,
                "category_id": new_cat,
                "bbox": ann["bbox"],
                "area": ann["area"],
                "iscrowd": ann.get("iscrowd", 0)
            })
            ann_idx += 1
            
        img_idx += 1

print("=== 2. Merging ScreenSpot Dataset ===")
add_coco_dataset("data/screenspot/train/images", "data/screenspot/train/annotations.json", val_ratio=0.0, dataset_prefix="screenspot")
add_coco_dataset("data/screenspot/val/images", "data/screenspot/val/annotations.json", val_ratio=1.0, dataset_prefix="screenspot")
print(f"ScreenSpot merged. Total images: {img_idx - 1}")

print("=== 3. Merging GroundUI-18k Dataset ===")
try:
    groundui = fouh.load_from_hub("Voxel51/GroundUI-18k")
    groundui.export(export_dir="data/groundui_coco", dataset_type=fo.types.COCODetectionDataset, label_field="detections")
    add_coco_dataset("data/groundui_coco/data", "data/groundui_coco/labels.json", val_ratio=0.15, dataset_prefix="groundui")
    print(f"GroundUI-18k merged. Total images: {img_idx - 1}")
except Exception as e:
    print(f"Error importing GroundUI-18k: {e}")

print("=== 4. Merging Rico Dataset (10,000 Subset) ===")
try:
    rico = fouh.load_from_hub("Voxel51/rico")
    # Take 10,000 subset to keep build times reasonable
    rico_subset = rico.limit(10000)
    rico_subset.export(export_dir="data/rico_coco", dataset_type=fo.types.COCODetectionDataset, label_field="detections")
    add_coco_dataset("data/rico_coco/data", "data/rico_coco/labels.json", val_ratio=0.15, dataset_prefix="rico")
    print(f"Rico subset merged. Total images: {img_idx - 1}")
except Exception as e:
    print(f"Error importing Rico: {e}")

with open(output_dir / "train" / "annotations.json", "w") as f:
    json.dump(merged_train, f)
with open(output_dir / "val" / "annotations.json", "w") as f:
    json.dump(merged_val, f)

print(f"=== Merge Finished! Train: {len(merged_train['images'])} | Val: {len(merged_val['images'])} ===")


## 4. Run Dataset Preparation and Merge

In [ ]:
!python prepare_all_datasets.py

## 5. Write Fine-Tuning Script

In [ ]:
%%writefile finetune_rtdetr_ui.py
import argparse
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision.datasets import CocoDetection
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor, Trainer, TrainingArguments

@dataclass(frozen=True)
class TrainPaths:
    train_images: Path
    train_annotations: Path
    val_images: Path
    val_annotations: Path

class CocoUIDataset(Dataset):
    def __init__(self, image_dir: Path, annotations_file: Path, processor: Any) -> None:
        self._base = CocoDetection(root=str(image_dir), annFile=str(annotations_file))
        self.processor = processor
    def __len__(self) -> int:
        return len(self._base)
    def __getitem__(self, idx: int) -> dict[str, object]:
        image, annotations = self._base[idx]
        image_id = self._base.ids[idx]
        encoding = self.processor(images=image, annotations={"image_id": image_id, "annotations": annotations}, return_tensors="pt")
        pixel_values = encoding["pixel_values"].squeeze(0)
        labels = encoding["labels"][0]
        return {"pixel_values": pixel_values, "labels": labels}

def collate_fn(batch: list[dict[str, Any]]) -> dict[str, Any]:
    pixel_values = [sample["pixel_values"] for sample in batch]
    labels = [sample["labels"] for sample in batch]
    return {"pixel_values": torch.stack(pixel_values), "labels": labels}

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train-images", type=Path, required=True)
    parser.add_argument("--train-annotations", type=Path, required=True)
    parser.add_argument("--val-images", type=Path, required=True)
    parser.add_argument("--val-annotations", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=3e-5)
    parser.add_argument("--save-total-limit", type=int, default=2)
    args = parser.parse_args()

    processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r18vd_coco_o365")
    model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r18vd_coco_o365")

    train_dataset = CocoUIDataset(args.train_images, args.train_annotations, processor)
    val_dataset = CocoUIDataset(args.val_images, args.val_annotations, processor)

    training_args = TrainingArguments(
        output_dir=str(args.output_dir),
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        weight_decay=1e-4,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        remove_unused_columns=False,
        save_strategy="epoch",
        evaluation_strategy="epoch",
        logging_steps=50,
        fp16=True, # GPU acceleration
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=args.save_total_limit,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
    )

    trainer.train()
    trainer.save_model(str(args.output_dir / "checkpoint-best"))
    processor.save_pretrained(str(args.output_dir / "checkpoint-best"))
    print("Training Complete. Model saved successfully!")

if __name__ == '__main__':
    main()


## 6. Execute Fine-Tuning

In [ ]:
!python finetune_rtdetr_ui.py \
    --train-images data/merged_coco/train/images \
    --train-annotations data/merged_coco/train/annotations.json \
    --val-images data/merged_coco/val/images \
    --val-annotations data/merged_coco/val/annotations.json \
    --output-dir checkpoints/rtdetr-ui-complete \
    --epochs 15 \
    --batch-size 8 \
    --lr 3e-5

## 7. Package and Export the Best Model Checkpoint

In [ ]:
!zip -r rtdetr-ui-complete-checkpoint-best.zip checkpoints/rtdetr-ui-complete/checkpoint-best